# Notebook 01 — Inferencia en HMMs

Implementamos Forward, Backward y Viterbi sobre el ejemplo de Lain/Clima.
Verificamos que los resultados coincidan con la traza manual del módulo.

## 1. Parámetros del ejemplo de Lain

In [ ]:
import numpy as np

# Estados: 0=Sol (S), 1=Lluvia (R)
# Observaciones: 0=sin paraguas, 1=con paraguas
PI  = np.array([0.6,  0.4])
A   = np.array([[0.7, 0.3],
                [0.4, 0.6]])
B   = np.array([[0.9, 0.1],
                [0.2, 0.8]])
OBS = np.array([0, 1, 1])   # sin paraguas, con, con
T, N = len(OBS), len(PI)
STATES = ['S (Sol)', 'R (Lluvia)']
print('Parámetros cargados.')
print(f'T={T} observaciones, N={N} estados')

## 2. Algoritmo Forward

In [ ]:
def forward(obs, pi, a, b):
    T_ = len(obs)
    alpha = np.zeros((T_, len(pi)))
    alpha[0] = pi * b[:, obs[0]]          # [P1] Inicialización
    for t in range(1, T_):                 # [P2] Recursión
        alpha[t] = (alpha[t-1] @ a) * b[:, obs[t]]
    return alpha, alpha[-1].sum()          # [P3] Terminación

alpha, p_obs = forward(OBS, PI, A, B)

print('Tabla de valores alpha:')
print(f"{'t':>3} {'O_t':>5} {'alpha(S)':>12} {'alpha(R)':>12} {'suma':>12}")
for t in range(T):
    s = alpha[t].sum()
    print(f'{t+1:>3} {OBS[t]:>5} {alpha[t,0]:>12.5f} {alpha[t,1]:>12.5f} {s:>12.5f}')
print(f'P(O|lambda) = {p_obs:.5f}')

ALPHA_EXP = np.array([[0.54000, 0.08000],
                       [0.04100, 0.16800],
                       [0.00959, 0.09048]])
assert np.allclose(alpha, ALPHA_EXP, atol=1e-4), 'Alpha no coincide'
assert abs(p_obs - 0.10007) < 1e-4
print('Verificación: coincide con la traza manual. ✓')

## 3. Algoritmo Backward

In [ ]:
def backward(obs, a, b):
    T_ = len(obs)
    beta = np.zeros((T_, a.shape[0]))
    beta[-1] = 1.0                                   # [P1] Inicialización
    for t in range(T_-2, -1, -1):                    # [P2] Recursión
        beta[t] = a @ (b[:, obs[t+1]] * beta[t+1])
    return beta

beta = backward(OBS, A, B)

print('Tabla de valores beta:')
print(f"{'t':>3} {'O_t':>5} {'beta(S)':>12} {'beta(R)':>12}")
for t in range(T):
    print(f'{t+1:>3} {OBS[t]:>5} {beta[t,0]:>12.4f} {beta[t,1]:>12.4f}')

p_obs_bwd = (PI * B[:, OBS[0]] * beta[0]).sum()
print(f'Verificación cruzada Backward: P(O|lambda) = {p_obs_bwd:.5f}')

BETA_EXP = np.array([[0.14650, 0.26200],
                      [0.31000, 0.52000],
                      [1.00000, 1.00000]])
assert np.allclose(beta, BETA_EXP, atol=1e-4)
assert abs(p_obs_bwd - 0.10007) < 1e-4
print('Forward == Backward. ✓')

## 4. Probabilidades posteriores γ_t(i)

In [ ]:
gamma = (alpha * beta) / p_obs

print('Probabilidades posteriores gamma:')
print(f"{'t':>3} {'O_t':>5} {'gamma(S)':>12} {'gamma(R)':>12} {'suma':>8}")
for t in range(T):
    print(f'{t+1:>3} {OBS[t]:>5} {gamma[t,0]:>12.4f} {gamma[t,1]:>12.4f} {gamma[t].sum():>8.4f}')

print('Interpretación:')
for t in range(T):
    est = STATES[gamma[t].argmax()]
    print(f'  t={t+1} (O={OBS[t]}): estado más probable = {est} ({gamma[t].max():.1%})')

## 5. Algoritmo de Viterbi

In [ ]:
def viterbi(obs, pi, a, b):
    T_ = len(obs)
    N_ = len(pi)
    delta = np.zeros((T_, N_))
    psi   = np.zeros((T_, N_), dtype=int)
    delta[0] = pi * b[:, obs[0]]                    # [P1] Inicialización
    psi[0]   = 0
    for t in range(1, T_):                          # [P2] Recursión
        for j in range(N_):
            scores     = delta[t-1] * a[:, j]
            psi[t,j]   = scores.argmax()
            delta[t,j] = scores[psi[t,j]] * b[j, obs[t]]
    path = np.zeros(T_, dtype=int)                  # [P3] Terminación
    path[-1] = delta[-1].argmax()
    for t in range(T_-2, -1, -1):                   # [P4] Backtracking
        path[t] = psi[t+1, path[t+1]]
    return path, delta, psi

path, delta, psi = viterbi(OBS, PI, A, B)

print('Tabla delta y backpointers:')
print(f"{'t':>3} {'O_t':>5} {'delta(S)':>12} {'psi(S)':>8} {'delta(R)':>12} {'psi(R)':>8}")
for t in range(T):
    ps = STATES[psi[t,0]] if t > 0 else '—'
    pr = STATES[psi[t,1]] if t > 0 else '—'
    print(f'{t+1:>3} {OBS[t]:>5} {delta[t,0]:>12.5f} {ps:>8} {delta[t,1]:>12.5f} {pr:>8}')

print(f"Secuencia óptima: {' → '.join(STATES[s] for s in path)}")
assert list(path) == [0, 1, 1]
print('Verificación: S → R → R ✓')

## 6. Visualización del trellis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
COLORS_MAP = {'S (Sol)': '#2E86AB', 'R (Lluvia)': '#E94F37'}
titles = ['Forward — α_t(i)', 'Viterbi — δ_t(i)']
mats   = [alpha, delta]

for ax_idx, (ax, mat, title) in enumerate(zip(axes, mats, titles)):
    ax.set_title(title, fontsize=12, fontweight='bold')
    ys = [1, 0]   # S en y=1, R en y=0
    for t in range(T):
        for s in range(N):
            col = '#2E86AB' if s == 0 else '#E94F37'
            ax.add_patch(plt.Circle((t, ys[s]), 0.20, color=col, alpha=0.75, zorder=3))
            ax.text(t, ys[s]+0.01, f'{mat[t,s]:.4f}', ha='center', va='center',
                    fontsize=7.5, color='white', fontweight='bold', zorder=4)
            ax.text(t, ys[s]-0.28, STATES[s], ha='center', fontsize=8, color='#2C3E50')
    for t in range(T-1):
        for si in range(N):
            for sj in range(N):
                ax.annotate('', xy=(t+1-0.21, ys[sj]), xytext=(t+0.21, ys[si]),
                           arrowprops=dict(arrowstyle='->', color='gray', alpha=0.4, lw=0.8))
    if ax_idx == 1:  # camino óptimo en verde
        for t in range(T-1):
            ax.annotate('', xy=(t+1-0.21, ys[path[t+1]]), xytext=(t+0.21, ys[path[t]]),
                       arrowprops=dict(arrowstyle='->', color='#27AE60', lw=2.5))
        ax.text(1, 1.40, 'Camino óptimo: S→R→R', ha='center',
                fontsize=9, color='#27AE60', fontweight='bold')
    ax.set_xlim(-0.5, T-0.5)
    ax.set_ylim(-0.6, 1.6)
    ax.set_xticks(range(T))
    ax.set_xticklabels([f't={t+1}, O={OBS[t]}' for t in range(T)])
    ax.set_yticks([])
    ax.grid(False)

plt.tight_layout()
plt.show()

## Resumen

| Algoritmo | P(O\|λ) | Secuencia óptima |
|-----------|:-------:|:----------------:|
| Forward   | 0.10007 | — |
| Backward (cruzado) | 0.10007 | — |
| Viterbi   | — | S → R → R |

Ambas formas de calcular P(O\|λ) coinciden. La secuencia óptima es S→R→R.